In [ ]:
!pip install groq chromadb gradio pillow ipywidgets gTTS pygame

In [ ]:
!pip install python-dotenv
import os
import chromadb
import gradio as gr
import pyttsx3
from groq import Groq
from PIL import Image
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
# groq key
load_dotenv("../.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Connect to Groq 
client = Groq(api_key=GROQ_API_KEY)

# Load local embedding model 
embed_model = SentenceTransformer("all-MiniLM-L6-v2")  

def get_embedding(text):
    return embed_model.encode(text).tolist()

# Setup local vector database
chroma = chromadb.PersistentClient(path="./memory_db")
collection = chroma.get_or_create_collection("patient_memories")

print("Setup complete!")

In [ ]:
# Add memories (one per item in the list)
memories = [
    "Ashwathy is your daughter. She lives in Kochi. She got married in 2015.",
    "Your husband Mohan loved cricket and watched every India match. He loved gardening.",
    "You worked as a school teacher for 30 years. Your favourite subject was Math.",
    "Your grandson Arjun is 8 years old. He loves football.",
    "You were born in Kozhikode and grew up near the old market. You loved eating biriyani.",
    "Your best friend is Sunita. You have known her since college. She visits every Onam.",
    "You have a dog named Titu. He is a golden retriever and loves to play fetch.",
    "Your favourite movie is Nadodikkatu and you used to watch it.",
]

print(f"You have {len(memories)} memories ready to store.")

In [ ]:
# Clear old memories first
existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])
    print("Cleared old memories.")

# Store each memory with its embedding
for i, memory in enumerate(memories):
    embedding = get_embedding(memory)   
    collection.add(
        documents=[memory],
        embeddings=[embedding],
        ids=[f"memory_{i}"]
    )
    print(f"Stored memory {i+1}: {memory[:60]}...")

print(f"\nAll {len(memories)} memories stored!")

In [ ]:
!pip install sentence-transformers

In [ ]:
def ask_memory_anchor(question):
    if not question.strip():
        return "Please type a question.", None

    question_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=2
    )
    relevant_memories = "\n".join(results["documents"][0])
    print(f"Memories used:\n{relevant_memories}\n")

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a warm, gentle assistant talking DIRECTLY to an elderly dementia patient. Use ONLY the memories provided. Always say 'your daughter', 'your husband', 'your job' — never say 'my'. Speak directly to the patient using 'you' and 'your'. Keep answers short, simple and comforting — 2 to 3 sentences. Never say I don't know."
            },
            {
                "role": "user",
                "content": f"Memories:\n{relevant_memories}\n\nQuestion: {question}"
            }
        ],
        temperature=0.5,
        max_tokens=200
    )

    answer = response.choices[0].message.content
    return answer, find_photo(question)

print("ask_memory_anchor() ready!")

In [ ]:
def find_photo(question):
    photos_dir = "data/photos"
    if not os.path.exists(photos_dir):
        return None

    keyword_map = {
        # Specific events first (before general people keywords)
        "daughter's wedding": "wedding",
        "ashwathy's wedding": "wedding",
        "daughter's marriage": "wedding",
        "wedding": "wedding",
        "marriage": "wedding",
        "married": "wedding",
        "working": "working",
        "teacher": "working",
        "school": "working",
        "job": "working",
        "work": "working",

        # People keywords after events
        "husband": "mohan",
        "mohan": "mohan",
        "daughter": "ashwathy",
        "ashwathy": "ashwathy",
        "kochi": "ashwathy",
        "girl in kochi": "ashwathy",
        "grandson": "arjun",
        "arjun": "arjun",
        "friend": "sunita",
        "sunita": "sunita",
    }

    question_lower = question.lower()

    for keyword, filename in keyword_map.items():
        if keyword in question_lower:
            for ext in [".jpg", ".jpeg", ".png"]:
                full_path = os.path.join(photos_dir, filename + ext)
                if os.path.exists(full_path):
                    return Image.open(full_path)
    return None

os.makedirs("data/photos", exist_ok=True)
print("Photo helper ready!")
     

In [ ]:
from gtts import gTTS
import pygame
import os

def speak(text):
    try:
        audio_path = "speech_output.mp3"
        
        tts = gTTS(text=text, lang="en", slow=False)
        tts.save(audio_path)
        
        pygame.mixer.init()
        pygame.mixer.music.load(audio_path)
        pygame.mixer.music.play()
        
        while pygame.mixer.music.get_busy():
            pygame.time.Clock().tick(10)
        
        pygame.mixer.music.unload()
        
    except Exception as e:
        print(f"Voice error: {e}")

speak("Hello, I am your memory assistant.")
print("Text-to-speech ready!")
        

In [ ]:
import gradio as gr
import threading

def full_pipeline(question):
    answer, photo = ask_memory_anchor(question)
    threading.Thread(target=speak, args=(answer,)).start()
    return answer, photo

def voice_pipeline(audio):
    try:
        if audio is None:
            return "No audio recorded", "Please record something first", None
        
        with open(audio, "rb") as f:
            transcription = client.audio.transcriptions.create(
                model="whisper-large-v3",
                file=f,
                response_format="text"
            )
        
        question = transcription
        print(f"Patient said: {question}")
        answer, photo = ask_memory_anchor(question)
        threading.Thread(target=speak, args=(answer,)).start()
        return question, answer, photo
    
    except Exception as e:
        print(f"Full error: {e}")
        return "Error", str(e), None

def add_memory(new_memory):
    if not new_memory.strip():
        return "Please type a memory first."
    try:
        embedding = get_embedding(new_memory)
        existing = collection.get()
        new_id = f"memory_{len(existing['ids'])}"
        collection.add(
            documents=[new_memory],
            embeddings=[embedding],
            ids=[new_id]
        )
        return f"Memory saved!: {new_memory}"
    except Exception as e:
        return f"Error: {e}"

with gr.Blocks(title="Memory Anchor") as app:

    gr.Markdown("# Memory Anchor")
    gr.Markdown("### A gentle AI assistant for dementia patients")
    gr.Markdown("---")

    with gr.Tabs():

        # Tab 1 - Type question
        with gr.Tab("Type Question"):
            with gr.Row():
                with gr.Column(scale=3):
                    question_input = gr.Textbox(
                        label="Ask a question",
                        placeholder='Try: "Who is Ashwathy?" or "What was my job?"',
                        lines=2
                    )
                    ask_button = gr.Button("Ask", variant="primary", size="lg")
                with gr.Column(scale=2):
                    photo_output1 = gr.Image(label="Related Photo", height=200)

            answer_output1 = gr.Textbox(label="Answer", lines=4, interactive=False)

            gr.Examples(
                examples=[
                    ["Who is Ashwathy?"],
                    ["Tell me about my husband"],
                    ["What was my job?"],
                    ["Tell me about my daughter's wedding"],
                    ["Where did I grow up?"],
                ],
                inputs=question_input
            )

            ask_button.click(fn=full_pipeline, inputs=question_input, outputs=[answer_output1, photo_output1])
            question_input.submit(fn=full_pipeline, inputs=question_input, outputs=[answer_output1, photo_output1])

        # Tab 2 - Speak question
        with gr.Tab("Speak Question"):
            gr.Markdown("### Click mic, speak, then click Stop — it will answer automatically")

            with gr.Row():
                with gr.Column(scale=3):
                    audio_input = gr.Audio(
                        sources=["microphone"],
                        type="filepath",
                        label="Click mic to record"
                    )
                with gr.Column(scale=2):
                    photo_output2 = gr.Image(label="Related Photo", height=200)

            heard_box = gr.Textbox(label="What I heard", interactive=False)
            answer_output2 = gr.Textbox(label="Answer", lines=4, interactive=False)

            audio_input.stop_recording(
                fn=voice_pipeline,
                inputs=audio_input,
                outputs=[heard_box, answer_output2, photo_output2]
            )

        # Tab 3 - Add memory
        with gr.Tab("Add Memory"):
            gr.Markdown("### Family members can add new memories here")
            gr.Markdown("Type a memory about the patient and click Save. The AI will use it immediately.")

            new_memory_input = gr.Textbox(
                label="Type a new memory",
                placeholder="Example: Ashwathy got a promotion at work in 2024.",
                lines=3
            )
            add_button = gr.Button("Save Memory", variant="primary", size="lg")
            add_status = gr.Textbox(label="Status", interactive=False)

            gr.Examples(
                examples=[
                    ["Ashwathy called yesterday and she sounded very happy."],
                    ["Your favourite food is Kerala sadya especially on Onam."],
                    ["You used to love singing Malayalam songs in the evening."],
                ],
                inputs=new_memory_input
            )

            add_button.click(fn=add_memory, inputs=new_memory_input, outputs=add_status)

app.launch(share=True, theme=gr.themes.Soft())
